In [2]:
#年运行时间
import os
import sys
# CaLRepo = os.environ.get("CaLRepo")
CaLRepo = '/home/zyq0416/workspace/CaL'
# print(CaLRepo)
sys.path.append(f"{CaLRepo}/utilities/")
import math
import json
import numpy as np
from pyCostEstimator import Cost_Estimator
from scipy.optimize import fsolve
import matplotlib.pyplot as plt
import pandas as pd  # 添加pandas库用于Excel操作

## construct cost indictor
construct_labour_cost_indictor=0.25
engineering_project_cost_indictor=0.175
TASC_multiplier=1.13
piping_integration_cost_indictor=0.05
#单次储能时间8小时
Single_run_time=8*3600

M_cao = 56e-3  # kg/mol
M_caoh2 = 74e-3  # kg/mol
parameters = dict()
flue_gas_composistion = dict()
flue_gas_composistion["co2"] = 0.1338
flue_gas_composistion["o2"] = 0.0384
flue_gas_composistion["n2"] = 0.6975
parameters["flue_gas_composition"] = flue_gas_composistion
parameters["isentropic_eff_mc"] = 0.88
parameters["t_isentropic_eff_mc"] = 0.92
parameters["mechanical_eff"] = 0.98   #机械效率
parameters["industrial_waste_heat_t"] =300 #℃
parameters["heat_transfer_loss_eff"] = 0.96
parameters["t_amb"] = 20   #环境温度
parameters["p_amb"] = 101325   #环境压力

parameters["p_bray_L"] = 7.5e6

parameters["p_water_supply_in"] = 2e5 
parameters["water_pressure_drop_rate"] = 100 #100Pa/m
parameters["water_pipe_length"] = 1000
parameters["water_pump_hydraulic_efficiency"] = 0.75
parameters["water_pump_mechanical_efficiency"] = 0.94

parameters["cao_conversion"] = 0.95  #氧化钙转化率
parameters["cao_purity"] = 0.98 #氢氧化钙含量
parameters["dehydrator_eff"] = 0.95   #脱水器效率
parameters["steam_pressure_loss_ratio"] = 0.01
parameters["convey_consumption"] = 10e3/100
parameters["storage_dehydrator_distance"] = 100

parameters["hydrator_eff"] = 0.95   #水合器器效率
parameters["p_bray_L_B"] = 7.5e6

inputs={}
inputs["p_bray_H"] = 17273582#优化变量1，热泵循环最高压力
inputs["p_bray_M"] = 12266517 #优化变量2，热泵循环中间压力
inputs["p_Dehy"] = 1e5 #变量4，反应器压力
inputs["Economic Model Selection"] = 2 #经济模型选择，1Tesio，2Nathan T
inputs["Compressor power limit"] = 200e6#功率界限，影响齿轮离心和滚筒离心模型的选取，单位W，桶式离心需体积流量
inputs["Turbine power limit"] = 35e6
inputs["Dehy_overheating_temperature"] = 20 #变量2，脱水反应器过热温度
inputs["min_temperature_exchange"] = 15
inputs["min_temperature_HEN"] = 15

inputs["p_bray_H_B"] = 30e6
inputs["p_bray_MH_B"] = 16217752.142109105
inputs["p_bray_ML_B"] = 12217752.142109105
inputs["p_Hydr"] = 1e5
inputs["Hydr_overheating_temperature"] = 40

inputs["Hydr_cao_in"]=440
inputs["T_X"] = 1


economic_inputs={}
#economic_inputs["limestone_price"]=70
#economic_inputs["calciner_cost_factor"]=1
economic_inputs["caoh2_unit_price"]=1500 #元/吨
economic_inputs["caoh2_price/life_rate"] = 30
economic_inputs["elec_price"]=0 #元/千瓦时  ##S6
economic_inputs["hot_price_h"] = 0.1542 #每千瓦时0.1542元，42.84元/吉焦
economic_inputs["elec_price_h"] = 1.0827
economic_inputs["Annual_cycle_count"] = 320
economic_inputs["operation_hours"] =economic_inputs["Annual_cycle_count"] * 8
# 在循环中设置 economic_inputs["operation_hours"] = Annual_cycle_count * 8
economic_inputs["discount_ratio"]=6/100 #8%
economic_inputs["operational_years"]=30
economic_inputs["operation_labour_cost_indictor"]=0.025/2#劳动力比例
#年运行时间为一般机组的一半
economic_inputs["maintain_cost_indictor"]=0.025/2#维护比例 0.025,
#年运行时间为一般机组的一半
results_list = []
# 循环年运行天数(250-320)
for days in range(1, 51):
    # 设置年运行小时数
    parameters["Store_electrical_power"] = days*1e6  # 机组容量
    cost = Cost_Estimator(parameters)
    # 运行模型
    results = cost.solve(inputs, economic_inputs)
    
    # 提取结果
    LCOE = results["LCOE"]
    IRR = results["IRR"]
    TPC = results["construction"]["total as-spent"] / 1e6  # 单位：百万RMB
    perTPC = results["construction"]["total as-spent"] / 1e6  /days# 单位：百万RMB/MW
    TAC = results["operation"]["total as-spent"] / 1e6  # 单位：百万RMB
    perTAC = results["operation"]["total as-spent"] / 1e6 /days # 单位：百万RMB/MW
    AI = results["Annual_operating_income"] / 1e6  # 单位：百万RMB
    AP = results["Annual_operating_profit"] / 1e6  # 单位：百万RMB
    
    # 添加到结果列表
    results_list.append({
        "Store_electrical_power": days,
        "LCOE": LCOE,
        "IRR": IRR,
        "TPC_MRMB": TPC,
        "perTPC_MRMB": perTPC,
        "TAC_MRMB": TAC,
        "perTAC_MRMB": perTAC,
        "AI_MRMB": AI,
        "AP_MRMB": AP
    })
    
    # 打印进度
    print(f"Completed: {days} days")

# 转换为DataFrame并保存到Excel
df = pd.DataFrame(results_list)
excel_filename = "Sensitivity GM.xlsx"
df.to_excel(excel_filename, index=False)
print(f"结果已保存到: {excel_filename}")

# ====== 结束添加 ======

Completed: 1 days
Completed: 2 days
Completed: 3 days
Completed: 4 days
Completed: 5 days
Completed: 6 days
Completed: 7 days
Completed: 8 days
Completed: 9 days
Completed: 10 days
Completed: 11 days
Completed: 12 days
Completed: 13 days
Completed: 14 days
Completed: 15 days
Completed: 16 days
Completed: 17 days
Completed: 18 days
Completed: 19 days
Completed: 20 days
Completed: 21 days
Completed: 22 days
Completed: 23 days
Completed: 24 days
Completed: 25 days
Completed: 26 days
Completed: 27 days
Completed: 28 days
Completed: 29 days
Completed: 30 days
Completed: 31 days
Completed: 32 days
Completed: 33 days
Completed: 34 days
Completed: 35 days
Completed: 36 days
Completed: 37 days
Completed: 38 days
Completed: 39 days
Completed: 40 days
Completed: 41 days
Completed: 42 days
Completed: 43 days
Completed: 44 days
Completed: 45 days
Completed: 46 days
Completed: 47 days
Completed: 48 days
Completed: 49 days
Completed: 50 days
结果已保存到: Sensitivity GM.xlsx
